# 第八讲：Agent 系统开发

**课程**：Python 进阶（计算机学院）
**教师**：孙青 / 欧阳元新
**平台**：CloudStudio + CodeBuddy

---

**本讲目标**：基于 LangChain 构建一个带 RAG 工具 + 多轮对话记忆的智能问答 Agent，并用 Gradio 部署为可交互应用。

**前置条件**：
- 第6-7讲 RAG 流水线（分块、向量化、ChromaDB、检索、生成）
- 第3讲 OpenAI 兼容 API 调用
- Ollama 本地服务已启动

## 本讲全景导航

| Part | 主题 | 关键技术 | Vibe Coding 工具 |
|---|---|---|---|
| Part 1 | Agent 核心概念 | Agent 定义 / 三大组件 / vs RAG | Spec-driven |
| Part 2 | 工具调用 | Function Calling / LangChain Tool | Prompt 工程延伸 |
| Part 3 | ReAct Agent 实战 | LangChain Agent + RAG 工具 + Gradio | Inline Edit + @文件 |
| Part 4 | 多 Agent 与协议 | MCP / A2A / ANP | 上下文工程 |
| Part 5 | Skill + 课程总结 | Skill 写法 / 八讲工具链回顾 | 全工具链回顾 |

## 环境准备

In [1]:
# 在终端运行 ollama serve 
# 安装依赖（首次运行需要）
%pip install -q langchain langchain-openai langchain-community \
    chromadb sentence-transformers gradio markitdown


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 太慢了，还是调api吧（）

# import requests

# resp = requests.get("http://localhost:11434/api/tags")
# data = resp.json()

# models = [m["name"] for m in data.get("models", [])]

# print("Ollama 可用模型:", models if models else "无模型，请先 ollama pull qwen2.5:7b-instruct-q4_K_M")

---
## Part 1：从 RAG 到 Agent

**受众**：零基础学生
**前置条件**：第6-7讲 RAG 流水线
**学习目标**：
1. 理解 RAG 的局限性
2. 掌握 Agent 的定义和三大组件
3. 理解 ReAct 和 Plan-and-Execute 两种范式

### 1.1 RAG 的局限性

回顾第6-7讲，我们搭建了一个 RAG 问答系统：

```
用户提问 → 检索文档 → 拼接 Prompt → LLM 生成回答
```

但 RAG 有三个"不能"：
- **不能做计算**：问"增长率是多少"只能返回原文，不会算
- **不能多步推理**：问"先查A，再用A算B"做不到
- **不能执行动作**：不能调 API、不能写文件、不能上网搜索

**Agent 解决了这些问题**：Agent = LLM + 工具 + 记忆 + 规划

### 1.2 Agent 的三大组件

| 组件 | 功能 | 类比 |
|---|---|---|
| **规划（Planning）** | 任务拆解、推理决策 | 大脑前额叶 |
| **工具（Tools）** | 执行具体动作 | 人的双手 |
| **记忆（Memory）** | 保存对话历史 | 人的海马体 |

```
Agent = LLM（大脑） + Tools（双手） + Memory（记忆） + Planning（规划）
```

### 1.3 ReAct 范式：边想边做

**ReAct = Reasoning + Acting**

每一步都遵循：`Thought → Action → Observation` 循环

```
Thought 1: 用户问了航天发射次数，我需要查报告。
Action 1:  rag_search("2024年中国航天发射次数")
Observation 1: 根据报告，2024年中国完成约70次发射...

Thought 2: 还需要算增长率，先查去年数据。
Action 2:  rag_search("2023年中国航天发射次数")
Observation 2: 2023年完成67次发射...

Thought 3: 用计算器算增长率。
Action 3:  calculator("(70-67)/67*100")
Observation 3: 4.48

Thought 4: 数据齐了，可以回答了。
Final Answer: 2024年完成约70次发射，同比增长约4.5%。
```

---
## Part 2：工具调用（Tool Use）

**受众**：零基础学生
**前置条件**：第3讲 OpenAI API 调用
**学习目标**：
1. 理解 Function Calling 协议
2. 掌握 LangChain 工具定义方式
3. 理解工具描述对 Agent 决策的影响

### 2.1 Function Calling 协议

**传统方式 vs Function Calling**：
- 传统：用户 → LLM → 纯文本（LLM 不知道有工具）
- Function Calling：用户 → LLM → "调用 search 工具" → 执行 → 结果回传 → 最终回答

**完整 5 步流程**：
1. 用户提问 + 工具定义 一起发给 LLM
2. LLM 判断需要调用工具，返回工具名 + 参数（JSON）
3. 程序执行对应函数
4. 结果回传给 LLM
5. LLM 基于工具结果生成最终回答

### 2.2 初始化 LLM

In [3]:
from langchain_openai import ChatOpenAI

# 使用 Ollama 本地服务（复用第6-7讲）
# llm = ChatOpenAI(
#     model="qwen2.5:7b-instruct-q4_K_M",
#     base_url="http://localhost:11434/v1",
#     api_key="ollama",
#     temperature=0,
# )

# 使用硅基流动免费模型
llm = ChatOpenAI(
    model="Qwen/Qwen3-8B",
    base_url="https://api.siliconflow.cn/v1",
    api_key="sk-vedrofottwzpxveidegfstxtmtjibwiknziaqenxyqexqkdx",
    temperature=0,
)

# 测试连接
response = llm.invoke("请用一句话介绍什么是 Agent。")
print(response.content)

/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




Agent是具备感知环境、自主决策和执行任务能力的智能实体，能够根据目标在复杂场景中独立行动并实现预期功能。


### 2.3 定义工具：@tool 装饰器

工具的三要素：
- **名称**：唯一标识符，LLM 在 Action 中引用
- **描述**：自然语言说明用途（是 LLM 决策的关键依据）
- **执行逻辑**：实际执行的 Python 函数

In [4]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """执行数学计算。当需要进行加减乘除、百分比、增长率等数值计算时使用此工具。

    Args:
        expression: 数学表达式，如 '100 * 1.15' 或 '(70 - 55) / 55 * 100'
    """
    try:
        allowed_names = {"abs": abs, "round": round, "min": min, "max": max}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return str(result)
    except Exception as e:
        return f"计算错误: {e}"

# 验证工具定义
print(f"工具名称: {calculator.name}")
print(f"工具描述: {calculator.description}")
print(f"参数 Schema: {calculator.args}")
print(f"测试调用: {calculator.invoke('(70 - 67) / 67 * 100')}")

工具名称: calculator
工具描述: 执行数学计算。当需要进行加减乘除、百分比、增长率等数值计算时使用此工具。

    Args:
        expression: 数学表达式，如 '100 * 1.15' 或 '(70 - 55) / 55 * 100'
参数 Schema: {'expression': {'title': 'Expression', 'type': 'string'}}
测试调用: 4.477611940298507


### 2.4 工具描述的"Prompt 工程"

**工具描述 = 写给模型看的使用说明书**

| 维度 | 坏描述 | 好描述 |
|---|---|---|
| 内容 | "搜索工具" | "从航天产业报告中检索相关段落。当用户提问涉及航天数据、发射次数时使用。" |
| 问题 | LLM 不知道何时该用 | LLM 清楚使用场景 |

**黄金法则**：说清功能 + 触发条件 + 输入输出 + 边界

### 2.5 工具绑定到 LLM

In [5]:
# 将工具绑定到 LLM
llm_with_tools = llm.bind_tools([calculator])

# 测试：提一个需要计算的问题
response = llm_with_tools.invoke("请计算 (70 - 67) / 67 * 100，保留两位小数")

# 检查 LLM 是否决定调用工具
if response.tool_calls:
    print("LLM 决定调用工具:")
    for tc in response.tool_calls:
        print(f"  工具: {tc['name']}")
        print(f"  参数: {tc['args']}")
else:
    print("LLM 直接回答:", response.content)

LLM 决定调用工具:
  工具: calculator
  参数: {'expression': '(70 - 67) / 67 * 100'}


---
## Part 3：ReAct Agent 从零构建

**受众**：零基础学生
**前置条件**：Part 1-2 概念、第6-7讲 RAG 流水线
**学习目标**：
1. 构建完整 RAG 工具
2. 用 LangChain 创建 ReAct Agent
3. 添加多轮对话记忆
4. 用 Gradio 构建交互界面

### 3.1 构建 RAG 知识库

复用第6-7讲的 RAG 流水线：加载文档 → 分块 → 向量化 → 存入 ChromaDB。

In [6]:
import os
from sentence_transformers import SentenceTransformer
import chromadb

# 初始化嵌入模型
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"嵌入模型加载完成，维度: {embed_model.get_sentence_embedding_dimension()}")

嵌入模型加载完成，维度: 384


In [7]:
# 加载并处理航天产业报告
from markitdown import MarkItDown

md_converter = MarkItDown()
report_path = "/workspace/实验八/航天产业报告.pdf"

if os.path.exists(report_path):
    result = md_converter.convert(report_path)
    report_text = result.text_content
    print(f"文档加载成功，总字符数: {len(report_text)}")
    print(f"前200字: {report_text[:200]}...")
else:
    print(f"文件不存在: {report_path}")
    print("请确保航天产业报告.pdf 在 实验八/ 目录下")

/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


文档加载成功，总字符数: 9427
前200字: [Table_Header]
| 行业点评报告●国防军工  |     |     |     |     |     |     | 2024年 |     | 02 月28 | 日   |     |
| ------------ | --- | --- | --- | --- | --- | --- | ----- | --- | ------ | --- | --- |

| [Table...


In [8]:
# 递归字符分块（复用第6-7讲逻辑）
def recursive_split(text, chunk_size=500, overlap=100):
    """递归字符分块，支持重叠窗口"""
    separators = ["\n\n", "\n", "。", "；", " "]
    chunks = []

    def split_by_separator(text, sep_idx=0):
        if len(text) <= chunk_size:
            if text.strip():
                chunks.append(text.strip())
            return
        if sep_idx >= len(separators):
            # 强制按 chunk_size 切分
            for i in range(0, len(text), chunk_size - overlap):
                chunk = text[i:i + chunk_size]
                if chunk.strip():
                    chunks.append(chunk.strip())
            return

        sep = separators[sep_idx]
        parts = text.split(sep)
        current = ""
        for part in parts:
            if len(current) + len(part) + len(sep) <= chunk_size:
                current += part + sep
            else:
                if current.strip():
                    chunks.append(current.strip())
                current = part + sep
        if current.strip():
            split_by_separator(current.strip(), sep_idx + 1)

    split_by_separator(text)
    return chunks

chunks = recursive_split(report_text, chunk_size=500, overlap=100)
print(f"分块完成，共 {len(chunks)} 个块")
print(f"示例块: {chunks[0][:200]}...")

分块完成，共 13 个块
示例块: [Table_Header]
| 行业点评报告●国防军工  |     |     |     |     |     |     | 2024年 |     | 02 月28 | 日   |     |
| ------------ | --- | --- | --- | --- | --- | --- | ----- | --- | ------ | --- | --- |...


In [9]:
# 向量化并存入 ChromaDB
chroma_client = chromadb.Client()  # 内存模式，演示用

# 删除已有同名 collection（重复运行时）
try:
    chroma_client.delete_collection("aerospace_report")
except:
    pass

collection = chroma_client.create_collection(
    name="aerospace_report",
    metadata={"description": "航天产业报告向量库"}
)

# 批量向量化并存储
batch_size = 50
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]
    embeddings = embed_model.encode(batch).tolist()
    ids = [f"chunk_{j}" for j in range(i, i+len(batch))]
    collection.add(
        documents=batch,
        embeddings=embeddings,
        ids=ids,
    )
    print(f"已存储 {min(i+batch_size, len(chunks))}/{len(chunks)} 个块")

print(f"\n向量库构建完成，共 {collection.count()} 条记录")

已存储 13/13 个块

向量库构建完成，共 13 条记录


### 3.2 封装 RAG 检索工具

In [10]:
@tool
def rag_search(query: str) -> str:
    """从航天产业报告中检索与用户问题相关的段落。
    当用户提问涉及航天产业数据、发射次数、市场规模、
    重点企业、政策动态、商业航天发展趋势等内容时使用此工具。

    Args:
        query: 用户的检索问题
    """
    query_embedding = embed_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
    )
    if results["documents"][0]:
        retrieved = "\n---\n".join(results["documents"][0])
        return f"检索到以下相关内容：\n{retrieved}"
    return "未在航天产业报告中找到相关信息。"

# 测试 RAG 工具
print(rag_search.invoke("2024年中国商业航天发展情况"))

检索到以下相关内容：
核心观点：
分析师
| [T able事_S件u：mm2 | a月ry2] 7       |                 |     |                          |     |                  |     |     |     | [李Ta良b le_Authors]  |     |
| ----------------- | -------------- | --------------- | --- | ------------------------ | --- | ---------------- | --- | --- | --- | ------------------- | --- |
|                   |                | 日，航天科技集团发布《2023 |     |                          |     | 年中国航天科技活动蓝皮书》，并规 |     |     |     |                     |     |
| 划                 | 2024年航天活动任务。2月 |                 |     | 26日，巴塞罗那世界移动通信大会（MWC）召开， |     |                  |     |     |     | ：010-80927657      |     |
| 美英等               | 10国发表联合声明，支持   |                 |     | 6G原则。                    |     |                  |     |     |     |                     |     |
：liliang_yj@chinastock.com.cn
|   2024年航天发射次数预计增长 |     |     |     | 49.3%，其中商业发射跃升 |     |     | 50%。《蓝皮书》预计我 |     |     |     |     |
| ------------------ | --- | --- |

In [11]:
# 组合工具列表
tools = [rag_search, calculator]

print("Agent 可用工具:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

Agent 可用工具:
  - rag_search: 从航天产业报告中检索与用户问题相关的段落。
    当用户提问涉及航天产业数据、发射次数、市场规模、
    重点企业、...
  - calculator: 执行数学计算。当需要进行加减乘除、百分比、增长率等数值计算时使用此工具。

    Args:
        expr...


### 3.3 构建 ReAct Agent

In [12]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

# ReAct Agent 的 Prompt 模板
REACT_PROMPT = PromptTemplate.from_template(
"""你是一个航天产业分析助手。请基于航天产业报告回答用户问题。

你可以使用以下工具：
{tools}

请严格按照以下格式回答（每一步都要包含 Thought、Action、Action Input）：

Question: 用户的问题
Thought: 分析问题，决定下一步行动
Action: 工具名称（必须是以下之一：{tool_names}）
Action Input: 工具的输入参数
Observation: 工具返回的结果
... （可重复 Thought/Action/Action Input/Observation）
Thought: 我已经有了足够信息
Final Answer: 最终回答

注意：
- 数值计算必须使用 calculator 工具，不要心算
- 所有事实性回答必须基于 rag_search 检索结果
- 如果检索不到信息，如实告知用户

开始！

Question: {input}
{agent_scratchpad}"""
)

# 创建 ReAct Agent
agent = create_react_agent(llm, tools, REACT_PROMPT)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True,
)

print("ReAct Agent 构建完成!")

ReAct Agent 构建完成!


### 3.4 测试 Agent

In [13]:
# 测试1：简单检索问题
result = agent_executor.invoke({
    "input": "2024年中国商业航天的发展趋势是什么？"
})
print("\n" + "="*60)
print("最终回答:", result["output"])



> Entering new AgentExecutor chain...




Thought: 用户的问题是关于2024年中国商业航天的发展趋势，需要从产业报告中提取相关政策、市场、技术或企业动态的信息。
Action: rag_search
Action Input: 2024年中国商业航天发展趋势检索到以下相关内容：
核心观点：
分析师
| [T able事_S件u：mm2 | a月ry2] 7       |                 |     |                          |     |                  |     |     |     | [李Ta良b le_Authors]  |     |
| ----------------- | -------------- | --------------- | --- | ------------------------ | --- | ---------------- | --- | --- | --- | ------------------- | --- |
|                   |                | 日，航天科技集团发布《2023 |     |                          |     | 年中国航天科技活动蓝皮书》，并规 |     |     |     |                     |     |
| 划                 | 2024年航天活动任务。2月 |                 |     | 26日，巴塞罗那世界移动通信大会（MWC）召开， |     |                  |     |     |     | ：010-80927657      |     |
| 美英等               | 10国发表联合声明，支持   |                 |     | 6G原则。                    |     |                  |     |     |     |                     |     |
：liliang_yj@chinastock.com.cn
|   2024年航天发射次数预计增长 |     | 

In [14]:
# 测试2：需要计算的问题
result = agent_executor.invoke({
    "input": "如果航天发射次数每年增长10%，从70次开始，3年后会达到多少次？"
})
print("\n" + "="*60)
print("最终回答:", result["output"])



> Entering new AgentExecutor chain...




Thought: 用户的问题涉及数值计算，需要使用calculator工具来计算增长后的发射次数。
Action: calculator
Action Input: 70 * (1.1)^3计算错误: unsupported operand type(s) for ^: 'float' and 'int'

Thought: 需要重新调整表达式以避免使用不支持的运算符^，改用乘法形式计算复利增长。
Action: calculator
Action Input: 70 * 1.1 * 1.1 * 1.193.17000000000002

Final Answer: 航天发射次数每年增长10%，从70次开始，3年后将达到约93.17次。由于发射次数为整数，实际可能为93次或94次，具体取决于增长方式是否为连续复利或按年度整数计算。

> Finished chain.

最终回答: 航天发射次数每年增长10%，从70次开始，3年后将达到约93.17次。由于发射次数为整数，实际可能为93次或94次，具体取决于增长方式是否为连续复利或按年度整数计算。


### 3.5 添加多轮对话记忆

In [15]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 存储不同 session 的历史
session_histories = {}

def get_session_history(session_id: str):
    if session_id not in session_histories:
        session_histories[session_id] = ChatMessageHistory()
    return session_histories[session_id]

# 更新 Prompt，加入对话历史支持
REACT_PROMPT_WITH_HISTORY = PromptTemplate.from_template(
"""你是一个航天产业分析助手。请基于航天产业报告回答用户问题。

你可以使用以下工具：
{tools}

请严格按照以下格式回答：

Question: 用户的问题
Thought: 分析问题，决定下一步行动
Action: 工具名称（必须是以下之一：{tool_names}）
Action Input: 工具的输入参数
Observation: 工具返回的结果
... （可重复）
Thought: 我已经有了足够信息
Final Answer: 最终回答

注意：
- 数值计算必须使用 calculator 工具
- 所有事实性回答必须基于 rag_search 检索结果
- 如果用户的问题引用了之前的对话内容，请结合对话历史理解

之前的对话历史：
{chat_history}

开始！

Question: {input}
{agent_scratchpad}"""
)

# 重建 Agent（支持历史）
agent_with_history = create_react_agent(llm, tools, REACT_PROMPT_WITH_HISTORY)
agent_executor_with_history = AgentExecutor(
    agent=agent_with_history,
    tools=tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True,
)

# 包装为支持对话历史的 Runnable
agent_with_memory = RunnableWithMessageHistory(
    agent_executor_with_history,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("带记忆的 Agent 构建完成!")

带记忆的 Agent 构建完成!


/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
# 多轮对话测试
config = {"configurable": {"session_id": "demo_session"}}

# 第一轮
print("="*60)
print("第一轮对话")
print("="*60)
r1 = agent_with_memory.invoke(
    {"input": "报告中提到了哪些航天企业？"},
    config=config,
)
print("\n回答:", r1["output"])

# 第二轮（引用第一轮内容）
print("\n" + "="*60)
print("第二轮对话（追问）")
print("="*60)
r2 = agent_with_memory.invoke(
    {"input": "这些企业中哪个最有发展潜力？为什么？"},
    config=config,
)
print("\n回答:", r2["output"])

第一轮对话


> Entering new AgentExecutor chain...




Thought: 用户询问报告中提到的航天企业，需要从报告中检索相关信息。
Action: rag_search
Action Input: 哪些航天企业检索到以下相关内容：
核心观点：
分析师
| [T able事_S件u：mm2 | a月ry2] 7       |                 |     |                          |     |                  |     |     |     | [李Ta良b le_Authors]  |     |
| ----------------- | -------------- | --------------- | --- | ------------------------ | --- | ---------------- | --- | --- | --- | ------------------- | --- |
|                   |                | 日，航天科技集团发布《2023 |     |                          |     | 年中国航天科技活动蓝皮书》，并规 |     |     |     |                     |     |
| 划                 | 2024年航天活动任务。2月 |                 |     | 26日，巴塞罗那世界移动通信大会（MWC）召开， |     |                  |     |     |     | ：010-80927657      |     |
| 美英等               | 10国发表联合声明，支持   |                 |     | 6G原则。                    |     |                  |     |     |     |                     |     |
：liliang_yj@chinastock.com.cn
|   2024年航天发射次数预计增长 |     |     |     | 49.3%，其中商业发射跃升 |     | 

### 3.6 上下文窗口管理

In [17]:
# 演示：对话历史的 Token 管理
from langchain_core.messages import trim_messages, HumanMessage, AIMessage

# 模拟一段较长的对话历史
long_history = []
for i in range(20):
    long_history.append(HumanMessage(content=f"这是第{i+1}轮的用户问题，关于航天产业的某个方面"))
    long_history.append(AIMessage(content=f"这是第{i+1}轮的AI回答，包含了详细的分析内容" * 3))

print(f"原始对话历史: {len(long_history)} 条消息")

# 方法1：滑动窗口（保留最近 N 轮）
recent_history = long_history[-10:]  # 保留最近5轮
print(f"滑动窗口后: {len(recent_history)} 条消息")

# 方法2：Token 截断
# trimmed = trim_messages(long_history, max_tokens=4000, token_counter=llm, strategy="last")
# print(f"Token 截断后: {len(trimmed)} 条消息")

print("\n提示：实际项目中，建议用 Token 截断 + 始终保留 system message 的策略")

原始对话历史: 40 条消息
滑动窗口后: 10 条消息

提示：实际项目中，建议用 Token 截断 + 始终保留 system message 的策略


### 3.7 Gradio 交互界面

将 Agent 部署为可交互的 Web 界面——这是本课程的最终交付物！

In [ ]:
import gradio as gr

SESSION_ID = "gradio_user"

def chat_fn(message, history):
    try:
        response = agent_with_memory.invoke(
            {"input": message},
            config={
                "configurable": {
                    "session_id": SESSION_ID
                }
            },
        )

        # 兼容不同版本 LangChain
        if isinstance(response, dict):
            if "output" in response:
                output = response["output"]
            elif "messages" in response:
                output = response["messages"][-1]
            elif "answer" in response:
                output = response["answer"]
            else:
                output = str(response)
        else:
            output = response

        # AIMessage -> str
        if hasattr(output, "content"):
            output = output.content

        return str(output)

    except Exception as e:
        import traceback
        traceback.print_exc()
        return f"错误 {e}"


demo = gr.ChatInterface(
    fn=chat_fn,

    chatbot=gr.Chatbot(
        height=500,
    ),

    textbox=gr.Textbox(
        placeholder="例如：2024年中国商业航天发展趋势是什么？",
        label="输入问题",
    ),

    title=" 航天产业智能问答助手",

    description="基于 LangChain Agent，支持多轮对话、知识库检索和数值计算。",

    examples=[
        "2024年中国商业航天的发展趋势是什么？",
        "报告中提到了哪些重点企业？",
        "商业航天市场规模有多大？",
        "如果市场规模年增长15%，5年后会达到多少？",
    ],

    autofocus=True,
    save_history=False,
)


demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.




> Entering new AgentExecutor chain...


Thought: 用户询问2024年中国商业航天的发展趋势，需要从产业报告中提取相关信息。
Action: rag_search
Action Input: 2024年中国商业航天的发展趋势
检索到以下相关内容：
[Table_Header]
| 行业点评报告●国防军工  |     |     |     |     |     |     | 2024年 |     | 02 月28 | 日   |     |
| ------------ | --- | --- | --- | --- | --- | --- | ----- | --- | ------ | --- | --- |
---
核心观点：
分析师
| [T able事_S件u：mm2 | a月ry2] 7       |                 |     |                          |     |                  |     |     |     | [李Ta良b le_Authors]  |     |
| ----------------- | -------------- | --------------- | --- | ------------------------ | --- | ---------------- | --- | --- | --- | ------------------- | --- |
|                   |                | 日，航天科技集团发布《2023 |     |                          |     | 年中国航天科技活动蓝皮书》，并规 |     |     |     |                     |     |
| 划                 | 2024年航天活动任务。2月 |                 |     | 26日，巴塞罗那世界移动通信大会（MWC）召开， |     |                  |     |     |     | ：010-80927657      |     |
| 

---
## Part 4：多 Agent 协作与通信协议

**受众**：零基础学生
**前置条件**：Part 3 单 Agent 构建
**学习目标**：
1. 理解多 Agent 协作的基本模式
2. 理解 MCP/A2A/ANP 三种协议的设计理念

### 4.1 多 Agent 协作模式

单 Agent 处理复杂任务时容易出错、上下文爆炸。多 Agent 分工协作可以解决这个问题。

| 模式 | 描述 | 适用场景 |
|---|---|---|
| 顺序协作 | A → B → C 流水线 | 步骤明确的流程 |
| 并行协作 | A 和 B 同时工作 | 独立子任务 |
| 监督协作 | Manager 分配任务给 Worker | 复杂项目管理 |
| 辩论协作 | 多个 Agent 给出不同观点 | 需要多视角分析 |

In [25]:
# 多 Agent 协作示例（概念演示）
# 注意：这是伪代码，展示多 Agent 的协作思路

class SimpleMultiAgent:
    """简化版多 Agent 协作示例"""

    def __init__(self, llm):
        self.llm = llm

    def researcher(self, topic):
        """研究员 Agent：负责搜索资料"""
        response = self.llm.invoke(
            f"你是一个研究员。请搜索并总结关于'{topic}'的关键信息。"
        )
        return response.content

    def analyst(self, research_data):
        """分析师 Agent：负责数据分析"""
        response = self.llm.invoke(
            f"你是一个数据分析师。请分析以下资料并给出关键洞察：\n{research_data}"
        )
        return response.content

    def writer(self, analysis):
        """撰写员 Agent：负责写报告"""
        response = self.llm.invoke(
            f"你是一个报告撰写员。请基于以下分析撰写一段简洁的报告摘要：\n{analysis}"
        )
        return response.content

    def run(self, topic):
        """顺序协作：研究 → 分析 → 撰写"""
        print("Step 1: 研究员搜索资料...")
        research = self.researcher(topic)
        print("Step 2: 分析师分析数据...")
        analysis = self.analyst(research)
        print("Step 3: 撰写员写报告...")
        report = self.writer(analysis)
        return report

# 运行多 Agent 协作
multi_agent = SimpleMultiAgent(llm)
report = multi_agent.run("中国商业航天2024年发展趋势")
print("\n最终报告:")
print(report)

Step 1: 研究员搜索资料...
Step 2: 分析师分析数据...
Step 3: 撰写员写报告...

最终报告:


**中国商业航天2024年关键洞察分析报告摘要**  

2024年，中国商业航天在政策支持、技术突破与市场拓展三方面加速发展，进入“规模化+专业化”新阶段。政策层面，《关于促进商业航天发展的指导意见》明确产业方向，国家航天局与市场监管总局的监管优化提升了行业透明度，但需关注政策细则落地速度及对新兴技术的潜在限制。市场规模预计突破500亿元，卫星互联网、商业发射及空间科学实验成为核心增长点，融资总额同比增长30%，资本更聚焦可重复使用火箭与低轨卫星组网，但需警惕泡沫化风险。技术领域，可重复使用火箭显著降低发射成本，卫星互联网加速覆盖偏远地区与应急通信场景，商业卫星制造实现自主化突破，但标准化与长期运营成本仍是挑战。国际合作深化，与SpaceX、欧洲航天局及“一带一路”国家的协作推动中国走向全球舞台，同时面临美国技术封锁与地缘政治竞争压力。产业链协同创新趋势明显，上游核心技术国产化、中游发射能力提升及下游服务多元化逐步形成闭环生态。人才培养与创新平台建设为行业提供持续动力，但需提升产学研转化效率。未来需在技术自主化、细分领域差异化竞争、国际规则参与及监管市场平衡间寻求突破，以应对政策不确定性、技术瓶颈和国际竞争，实现从“跟跑”到“领跑”的跨越。


### 4.2 通信协议：MCP / A2A / ANP

| 协议 | 提出者 | 解决什么 | 类比 |
|---|---|---|---|
| **MCP** | Anthropic | Agent ↔ 工具通信 | USB-C 接口 |
| **A2A** | Google | Agent ↔ Agent 通信 | 对讲机 |
| **ANP** | 开源社区 | 大规模 Agent 网络 | 互联网路由 |

**MCP（Model Context Protocol）**：
- 统一了 Agent 访问外部工具的方式
- 已有 1000+ MCP Server 可用（GitHub、Slack、数据库...）
- 我们课程中用的 Proma 就大量使用了 MCP

**A2A（Agent-to-Agent Protocol）**：
- 让不同 Agent 直接对话协作
- 核心概念：Agent Card（名片）、Task（任务）、Message（消息）

**ANP（Agent Network Protocol）**：
- 在大规模网络中发现和路由 Agent 服务
- 目前仍在概念阶段

In [ ]:
# MCP 概念演示：展示 MCP 工具的标准化接入方式
# 注意：这是概念代码，展示 MCP 的设计思路

print("=== MCP 概念演示 ===")
print()
print("传统方式（每个服务写一个适配器）：")
print("  Agent → 自定义 WeatherTool 类 → 天气 API")
print("  Agent → 自定义 DatabaseTool 类 → 数据库")
print("  Agent → 自定义 GitHubTool 类 → GitHub API")
print()
print("MCP 方式（标准化协议）：")
print("  Agent → MCP Client → MCP Server（天气）")
print("  Agent → MCP Client → MCP Server（数据库）")
print("  Agent → MCP Client → MCP Server（GitHub）")
print()
print("MCP 三层架构：")
print("  Host（宿主）: 管理对话流程，如 Claude Desktop")
print("  Client（客户端）: 与 Server 通信")
print("  Server（服务器）: 提供具体工具能力")
print()
print("实际使用示例：")
print("  配置 mcp.json → 自动发现 MCP Server → Agent 获得新工具")

=== MCP 概念演示 ===

传统方式（每个服务写一个适配器）：
  Agent → 自定义 WeatherTool 类 → 天气 API
  Agent → 自定义 DatabaseTool 类 → 数据库
  Agent → 自定义 GitHubTool 类 → GitHub API

MCP 方式（标准化协议）：
  Agent → MCP Client → MCP Server（天气）
  Agent → MCP Client → MCP Server（数据库）
  Agent → MCP Client → MCP Server（GitHub）

MCP 三层架构：
  Host（宿主）: 管理对话流程，如 Claude Desktop / Proma
  Client（客户端）: 与 Server 通信
  Server（服务器）: 提供具体工具能力

实际使用示例：
  配置 mcp.json → 自动发现 MCP Server → Agent 获得新工具


---
## Part 5：Skill 写法 + 课程总结

**受众**：零基础学生  
**前置条件**：Part 1-4  
**学习目标**：
1. 以随课 `第8讲/skills/` 目录中的真实模板和示例理解 Skill
2. 能从 `skills/template/SKILL.md` 复制并改写一个与 Agent 一致的 `SKILL.md`
3. 用测试提示词检查 Skill 的触发条件、工具边界和输出要求
4. 回顾八讲 Vibe Coding 工具链全景


### 5.1 什么是 Skill？

**Skill = 以 `SKILL.md` 为入口的可复用 Agent 能力包**。它不是临时写在 Python 字典里的 Prompt，而是一个独立文件夹：根目录的 `SKILL.md` 用 YAML frontmatter 声明触发条件，并用 Markdown 说明工作流程、工具边界、输出和验证。

本讲的 Skill 内容请以随课 `第8讲/skills/` 文件夹为准：

| 先看什么 | 路径（相对 `第8讲/`） | 用途 |
|---|---|---|
| 技能库导航 | `skills/README_zh.md` | 了解模板、示例和规范入口 |
| 最小模板 | `skills/template/SKILL.md` | 复制后创建自己的 Skill |
| 创建与评估方法 | `skills/skills/skill-creator/SKILL.md` | 明确意图、编写、测试、迭代 |
| MCP 设计案例 | `skills/skills/mcp-builder/SKILL.md` | 观察如何约束工具接口 |
| 规范入口 | `skills/spec/agent-skills-spec.md` | 查阅 Agent Skills 规范链接 |

**课堂原则**：参考模板和示例的结构、验证方式；不要照搬与自己 Agent 无关的工具或业务流程。


In [1]:
# 读取随课 Skills 模板，并构造一个可检查的 SKILL.md 草稿
from pathlib import Path
import re

def find_lesson_dir(start: Path) -> Path:
    """无论从课程根目录还是第8讲目录启动 Notebook，都定位到第8讲。"""
    for base in (start.resolve(), *start.resolve().parents):
        for candidate in (base, base / "第8讲"):
            if (candidate / "skills" / "template" / "SKILL.md").is_file():
                return candidate
    raise FileNotFoundError("未找到 第8讲/skills/template/SKILL.md；请从课程资料v3或第8讲目录启动 Notebook。")

lesson_dir = find_lesson_dir(Path.cwd())
template_path = lesson_dir / "skills" / "template" / "SKILL.md"
creator_path = lesson_dir / "skills" / "skills" / "skill-creator" / "SKILL.md"

print("模板文件:", template_path)
print("创建方法示例:", creator_path)
print("\n--- 随课最小模板 ---")
print(template_path.read_text(encoding="utf-8"))

# 课堂草稿：真实交付时请保存为 实验八/skills/aerospace-analyst/SKILL.md
skill_draft = """---
name: aerospace-analyst
description: 在用户需要基于给定航天产业报告完成分析，且可能需要检索或计算时使用。
---

# Aerospace Analyst

## 工作流程
1. 判断问题是否需要报告中的证据。
2. 使用 rag_search 检索相关段落；只有必要时才使用 calculator。
3. 核对证据与计算结果；无证据时明确说明。

## 工具边界
- 允许：rag_search、calculator。
- 禁止：编造报告中不存在的数据；执行未授权的外部操作。

## 输出要求
- 依次给出：分析结论、数据依据、来源说明。
"""

frontmatter = re.match(r"^---\n(?P<meta>.*?)\n---", skill_draft, flags=re.S)
assert frontmatter, "SKILL.md 必须以 YAML frontmatter 开始"
metadata = dict(line.split(": ", 1) for line in frontmatter.group("meta").splitlines())
assert re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", metadata["name"]), "name 使用小写连字符格式"
assert metadata.get("description"), "description 必须说明何时使用该 Skill"
assert all(section in skill_draft for section in ["## 工作流程", "## 工具边界", "## 输出要求"])

print("--- 草稿检查通过 ---")
print("name:", metadata["name"])
print("description:", metadata["description"])
print("\n--- 自定义 Skill 草稿 ---")
print(skill_draft)


模板文件: /Users/jafekin/Documents/Python进阶课课件及案例开发/课程资料v3/第8讲/skills/template/SKILL.md
创建方法示例: /Users/jafekin/Documents/Python进阶课课件及案例开发/课程资料v3/第8讲/skills/skills/skill-creator/SKILL.md

--- 随课最小模板 ---
---
name: template-skill
description: Replace with description of the skill and when Claude should use it.
---

# Insert instructions below

--- 草稿检查通过 ---
name: aerospace-analyst
description: 在用户需要基于给定航天产业报告完成分析，且可能需要检索或计算时使用。

--- 自定义 Skill 草稿 ---
---
name: aerospace-analyst
description: 在用户需要基于给定航天产业报告完成分析，且可能需要检索或计算时使用。
---

# Aerospace Analyst

## 工作流程
1. 判断问题是否需要报告中的证据。
2. 使用 rag_search 检索相关段落；只有必要时才使用 calculator。
3. 核对证据与计算结果；无证据时明确说明。

## 工具边界
- 允许：rag_search、calculator。
- 禁止：编造报告中不存在的数据；执行未授权的外部操作。

## 输出要求
- 依次给出：分析结论、数据依据、来源说明。



### 5.2 从模板到可用 Skill：测试与课程总结

写完 `SKILL.md` 后，按 `skills/skills/skill-creator/SKILL.md` 的思路至少测试三类提示：
1. **应触发**：`请根据这份航天产业报告比较 2023 和 2024 年的发射情况。`
2. **需要工具边界**：`请用报告数据计算增长率，并标明数据来源。`
3. **不应编造**：`报告没有给出的 2030 年发射次数是多少？`（应明确说明缺少证据）

| 讲次 | Vibe Coding 工具 | 核心能力 |
|---|---|---|
| 第1讲 | Prompt 迭代、Rules、上下文工程 | 与 AI 高效对话的基础 |
| 第2讲 | Spec-driven（SpeckitList）、MCP | 用规约管理 AI 行为 |
| 第3讲 | System Prompt、上下文工程 | 控制模型输出的方向 |
| 第4讲 | @文件引用、Inline Edit | 让 AI 理解并修改代码 |
| 第6-7讲 | Spec-driven 部署、Inline Edit 调参 | 用 AI 管理工程项目 |
| 第8讲 | Agent Skills、MCP 协议 | 用 `SKILL.md` 封装、测试和复用 Agent 能力 |

**演进逻辑**：
- 第1-3讲：**人驱动 AI**（人写 Prompt，AI 执行）
- 第4-5讲：**人 AI 协作**（AI 理解代码上下文，辅助修改）
- 第8讲：**受边界约束的 AI 行动**（Agent 调用工具；Skill 使工作流、边界和验证可复用）


---
## 知识点速查表

| 概念 | 一句话解释 |
|---|---|
| Agent | 感知 → 决策 → 行动 → 观察的闭环系统 |
| ReAct | Thought-Action-Observation 的推理—行动循环 |
| Plan-and-Execute | 先规划完整计划，再逐步执行 |
| Function Calling | LLM 返回“调哪个工具 + 什么参数”的结构化请求 |
| Tool Description | 写给模型看的工具使用说明，决定调用准确率 |
| Agent Memory | 短期（对话历史）+ 长期（向量化存储） |
| MCP | Agent ↔ Tool 的标准通信协议（USB-C） |
| A2A | Agent ↔ Agent 的点对点协作协议（对讲机） |
| Skill | 以 `SKILL.md` 描述触发、工作流、工具边界、输出与验证的能力包 |


## Vibe Coding 工具速查表

| 工具 | 用途 | 本讲可操作的材料 |
|---|---|---|
| Prompt 迭代 | 优化 AI 输出 | 反复修改 Prompt |
| Rules | 设定基本规矩 | 项目根目录 rules 文件 |
| Spec-driven | 锁定行为边界 | speckit.md + @引用 |
| System Prompt | 控制角色方向 | API 调用中 system 字段 |
| 上下文工程 | 组织输入信息 | 精心排列 messages |
| @文件引用 | 注入文件内容 | @ + 文件路径 |
| Inline Edit | 局部修改代码 | Cmd+K 选中代码块 |
| Agent Skills | 定义、测试和复用 Agent 能力 | `skills/template/SKILL.md` + `skills/skills/skill-creator/SKILL.md` |
| MCP 协议 | 标准化工具接入 | `skills/skills/mcp-builder/SKILL.md` |
